# Generating QRC inputs from ORCA 6 frequency calculations

pyQRC reads a completed frequency calculation and writes a new input file whose geometry has been displaced along one or more normal modes — Silva and Goodman's *Quick Reaction Coordinate* (QRC) approach. This notebook walks through the ORCA 6 example files that ship in this directory.

Requirements: `pip install pyqrc` (pulls in cclib and numpy).

> **Note:** parsing ORCA 6 outputs needs a newer cclib than the current PyPI release (1.8.1). Install cclib from GitHub master alongside pyQRC:
> ```
> pip install pyqrc
> pip install --upgrade git+https://github.com/cclib/cclib.git
> ```
> See the README “ORCA 6 compatibility” section.


## Setup

pyQRC writes its new input files next to the file it reads, so we copy the example outputs into a `scratch/` subdirectory and run everything there. The helper below invokes the same `pyqrc` command line you would use in a terminal or HPC batch script.

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()                # this examples directory
SCRATCH = HERE / "scratch"
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()

for name in ['acetaldehyde.out', 'claisen_ts.out']:
    shutil.copy(HERE / name, SCRATCH / name)


def run_pyqrc(*args):
    """Run the pyqrc command line inside the scratch directory."""
    result = subprocess.run(
        [sys.executable, "-m", "pyqrc", *args],
        cwd=SCRATCH, capture_output=True, text=True,
    )
    print(result.stdout, end="")
    if result.returncode != 0:
        print(result.stderr, end="")
        raise RuntimeError(f"pyqrc exited with code {result.returncode}")


def show(filename, n=40):
    """Print up to n lines of a file in the scratch directory."""
    lines = (SCRATCH / filename).read_text().splitlines()
    print("\n".join(lines[:n]))
    if len(lines) > n:
        print(f"... ({len(lines) - n} more lines)")


## Example 1: remove an unwanted imaginary frequency

This acetaldehyde optimization inadvertently produced a saddle point — it has one small imaginary frequency. By default pyQRC displaces along **all** imaginary modes, which is exactly what we want here: the displaced geometry breaks the symmetry of the saddle point, and re-optimizing it gives the true minimum.

In [2]:
run_pyqrc("acetaldehyde.out", "--nproc", "4", "--mem", "8GB")

o   acetaldehyde.out had 1 imaginary frequencies: processed


That wrote two files: `acetaldehyde_QRC.inp` (the new, displaced input — ready to submit) and `acetaldehyde_QRC.qrc` (a human-readable summary of the frequencies and the displacement).

In [3]:
show("acetaldehyde_QRC.inp")

! M062X D3Zero def2-TZVP Opt Freq
 %pal nprocs 4 end
 %maxcore 8192

# acetaldehyde_QRC

* xyz 0 1
 C   0.23618700   0.41131180  -0.01736280
 O   1.21130820  -0.28571240   0.01478220
 H   0.34318860   1.51327080  -0.08253560
 C  -1.16787080  -0.13484440   0.00549760
 H  -1.91375560   0.65682480   0.10307740
 H  -1.34180580  -0.68144120  -0.92950220
 H  -1.26293380  -0.85012200   0.82905400
*


In [4]:
show("acetaldehyde_QRC.qrc", n=24)

 pyQRC - a quick alternative to IRC calculations
 version: 2.3.0 / author: Robert Paton / email: robert.paton@colostate.edu
 Based on: Goodman, J. M.; Silva, M. A. Tet. Lett. 2003, 44, 8233-8236;
 Tet. Lett. 2005, 46, 2067-2069.

                -----ORIGINAL GEOMETRY------
                       X         Y         Z
   C            0.236194  0.411275  0.001351
   O            1.211305 -0.285687 -0.000292
   H            0.343212  1.513360 -0.001250
   C           -1.167871 -0.134852  0.000269
   H           -1.913641  0.656933  0.000318
   H           -1.302003 -0.765268 -0.879506
   H           -1.302906 -0.766368  0.879104

                ----HARMONIC FREQUENCIES----
                    Freq  Red mass   F const
               -190.7400    0.0000    0.0000
                515.1100    0.0000    0.0000
                752.4700    0.0000    0.0000
                946.0200    0.0000    0.0000
               1103.5600    0.0000    0.0000
               1144.7100    0.0000    0.0000
    

## Example 2: map a reaction coordinate (the namesake QRC)

For a transition state — here a Claisen rearrangement — the quick alternative to an IRC is two displaced inputs: one along the imaginary mode (`--amp 0.3`) and one in the reverse direction (`--amp -0.3`). Optimizing both gives the reactant and product the TS connects. The benchmark in the README found an amplitude of **0.3** performs best, hence the values used here; `--name` controls the suffix of the generated files.

In [5]:
run_pyqrc("claisen_ts.out", "--nproc", "4", "--mem", "8GB", "--amp", "0.3", "--name", "QRCF")
run_pyqrc("claisen_ts.out", "--nproc", "4", "--mem", "8GB", "--amp", "-0.3", "--name", "QRCR")

o   claisen_ts.out had 1 imaginary frequencies: processed
o   claisen_ts.out had 1 imaginary frequencies: processed


In [6]:
show("claisen_ts_QRCF.inp", n=14)
print("=" * 60)
show("claisen_ts_QRCR.inp", n=14)

! wB97X-D3 def2-SVP OptTS Freq
 %pal nprocs 4 end
 %maxcore 8192

# claisen_ts_QRCF

* xyz 0 1
 C  -1.29903370   0.85232150  -0.25975320
 C  -1.26552080  -0.49517880   0.26246170
 O  -0.59069340  -1.36466300  -0.24766300
 C   1.40091060  -0.85824280   0.19416940
 C   1.36559110   0.38855000  -0.30341050
 C   0.49374640   1.38058420   0.29401600
 H  -2.02027740   1.54638730   0.19052240
... (8 more lines)
! wB97X-D3 def2-SVP OptTS Freq
 %pal nprocs 4 end
 %maxcore 8192

# claisen_ts_QRCR

* xyz 0 1
 C  -1.51985230   0.74088050  -0.30567480
 C  -1.26975920  -0.43770720   0.26734630
 O  -0.37303860  -1.32175700  -0.21863500
 C   1.11343740  -0.94951720   0.15407860
 C   1.35043090   0.43425800  -0.30220150
 C   0.73957360   1.43039980   0.33589600
 H  -2.18053260   1.47074470   0.16898360
... (8 more lines)


## Using the Python API instead of the CLI

The same machinery is importable. `QRCGenerator` parses the output, computes the displaced geometry, and (unless `write=False`) writes the files in one go. With `write=False` you can inspect the displacement before committing anything to disk.

In [7]:
import numpy as np
from pyqrc import QRCGenerator

qrc = QRCGenerator(
    file=str(SCRATCH / "claisen_ts.out"),
    amplitude=0.3,
    nproc=4,
    mem="8GB",
    route=None,     # None clones the route/keywords from the original job
    verbose=False,  # skip the .qrc summary file
    suffix="API",
    val=None,       # or displace along the mode nearest this frequency (cm-1)
    num=None,       # or along this 1-indexed mode number
    write=False,    # compute only; no files are written
)

freqs = np.asarray(qrc.FREQS)
print("Imaginary frequencies (cm-1):", freqs[freqs < 0.0])
print(f"Mass-weighted displacement from the TS: {qrc.MW_DISTANCE:.4f} bohr amu^1/2")
print("Displaced geometry has clashing atoms:", qrc.OVERLAPPED)
print()

per_atom = np.linalg.norm(qrc.NEW_CARTESIAN - qrc.CARTESIAN, axis=1)
print("Atoms that move the most:")
for i in np.argsort(per_atom)[::-1][:5]:
    print(f"  atom {i + 1:>2} ({qrc.ATOMTYPES[i]:<2}) moved {per_atom[i]:.3f} Angstrom")


Imaginary frequencies (cm-1): [-636.62]
Mass-weighted displacement from the TS: 1.7932 bohr amu^1/2
Displaced geometry has clashing atoms: False

Atoms that move the most:
  atom  4 (C ) moved 0.152 Angstrom
  atom  6 (C ) moved 0.127 Angstrom
  atom  1 (C ) moved 0.126 Angstrom
  atom  3 (O ) moved 0.112 Angstrom
  atom  7 (H ) moved 0.089 Angstrom


## Where the files went

Everything generated above is in the `scratch/` subdirectory (ignored by git) — delete it when you are done. In real use you would submit the new input file (`*.inp`) to your scheduler and optimize.

See the [project README](../../README.md) for the full option list and for the IRC-comparison benchmark behind the recommended amplitude of 0.3.
